Question 1
rpk version (Redpanda CLI): v25.3.10 (rev 534e36666feb4a9bf7576bebc3290fc849dd7141)

Question 2
TOPIC        STATUS
green-trips  OK

Question 3 
True

Question 4
took 110.10918474197388 seconds to send 476386 messages

In [4]:
import json

from kafka import KafkaProducer

topic_name = 'green-trips'

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

server = 'localhost:19092'


producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

producer.bootstrap_connected()

True

In [5]:
import pandas as pd
import sys

sys.path.append("../src")
from models import Ride, ride_from_row, ride_serializer

df = pd.read_parquet("../src/data/green_tripdata_2025-10.parquet")

df = df[
    [
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "lpep_pickup_datetime",
        "lpep_dropoff_datetime",
        "passenger_count",
        "tip_amount",
    ]
]
df.head()

,PULocationID,DOLocationID,trip_distance,lpep_pickup_datetime,lpep_dropoff_datetime,passenger_count,tip_amount
0,247,69,0.70,2025-10-01 00:21:47,2025-10-01 00:24:37,1.0,1.70
1,66,25,1.61,2025-10-01 00:14:03,2025-10-01 00:24:14,1.0,2.78
2,244,244,0.00,2025-10-01 00:16:44,2025-10-01 00:16:47,1.0,2.20
3,95,170,10.37,2025-10-01 00:07:36,2025-10-01 00:32:14,1.0,11.31
4,82,138,4.07,2025-09-30 21:10:29,2025-09-30 21:22:30,1.0,6.82


In [6]:
from time import time
import json
t0 = time()
for index, row in df.iterrows():
    message = row.to_dict()
    message['lpep_pickup_datetime'] = str(message['lpep_pickup_datetime'])
    message['lpep_dropoff_datetime'] = str(message['lpep_dropoff_datetime'])
    producer.send(topic_name, message)

producer.flush()

t1 = time()
took = t1 - t0
print(f"took {took} seconds to send {len(df)} messages")

took 7.301268100738525 seconds to send 49416 messages
